# WORKHOUSE MCE: H0 closure cap 216 and G3 rerun readiness

This notebook is pinned to the August 22 frozen WORKHOUSE commit. It creates a **derived** engine with the one-line `max_states: 100 -> 216` revision, leaving the imported evidence untouched.

It then runs:

1. exact base/candidate SHA-256 checks;
2. the exhaustive target-free H0 resource audit over all 243,288 raw R2 seeds;
3. the original 47/47 engine self-tests;
4. the 609-evaluation zero-physics geometry preflight;
5. a fail-closed G3 output-sufficiency gate.

> **Safe default:** the current cap-only engine is expected to pass steps 1–4 and fail step 5 because it discards concrete endpoint displacement before checkpointing. The production cells stay disabled until a separately reviewed engine preserves and seals the endpoint-resolved H4 kernel.


In [ ]:
from pathlib import Path
import hashlib, json, os, re, shutil, subprocess, sys, time

REPO_URL = 'https://github.com/ats314/WORKHOUSE.git'
FROZEN_COMMIT = 'bcddd4ac5d53c0aad120b7784e46c29f15c2aed2'
BASE_SHA256 = 'be9d77f5b245715ed6e4fe6dc9178a56ddfa5c68efe697eaa7cf4bb6adae27ad'
CAP_ONLY_SHA256 = '85940912910b18cf4c530daa638722f6210dd6bce4a50dde1456e5b0012b5e19'
COVERAGE_SHA256 = '4e7f5acfd5610a2bd434e88f94c6ba2ba12a258e618a1249f49472f76c5dbd73'
PREFLIGHT_SHA256 = '576a4a3f00a41f1805fd015836107fb27ebc44190bd57629c13c17cc28e9f16f'

# 'cap_only' creates the reviewed one-line derivative.
# 'upload' accepts a separately reviewed G3-capable engine and requires its SHA below.
CANDIDATE_MODE = 'cap_only'
REVIEWED_UPLOAD_SHA256 = ''

ROOT = Path('/content/WORKHOUSE')
DERIVED = Path('/content/WORKHOUSE_MCE_DERIVED')
ENGINE_REL = Path('corpus-import/programs/hodge_o4_adjudication/src/DATA_SU3_Exact_MarkedCluster_m4_Colab.py')
HARNESS_REL = Path('settlement/mce_adjudication_harness.py')

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

print('Configuration loaded. Candidate mode:', CANDIDATE_MODE)


In [ ]:
if not (ROOT / '.git').is_dir():
    subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout', REPO_URL, str(ROOT)], check=True)
subprocess.run(['git', '-C', str(ROOT), 'sparse-checkout', 'init', '--cone'], check=True)
subprocess.run([
    'git', '-C', str(ROOT), 'sparse-checkout', 'set',
    'corpus-import/programs/hodge_o4_adjudication', 'settlement'
], check=True)
subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', FROZEN_COMMIT], check=True)
subprocess.run(['git', '-C', str(ROOT), 'checkout', '--detach', FROZEN_COMMIT], check=True)

base_engine = ROOT / ENGINE_REL
harness = ROOT / HARNESS_REL
assert base_engine.is_file() and harness.is_file()
assert sha256_file(base_engine) == BASE_SHA256, sha256_file(base_engine)
print('Frozen commit:', subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip())
print('Base engine SHA-256:', sha256_file(base_engine))


In [ ]:
DERIVED.mkdir(parents=True, exist_ok=True)
candidate_engine = DERIVED / base_engine.name

if CANDIDATE_MODE == 'cap_only':
    raw = base_engine.read_bytes()
    old = b'def closure(seed_state: State, max_states: int = 100) -> list[State]:'
    new = b'def closure(seed_state: State, max_states: int = 216) -> list[State]:'
    assert raw.count(old) == 1 and raw.count(new) == 0
    revised = raw.replace(old, new)
    assert len(revised) == len(raw) == 288_783
    candidate_engine.write_bytes(revised)
    expected_candidate_sha = CAP_ONLY_SHA256
elif CANDIDATE_MODE == 'upload':
    from google.colab import files
    uploaded = files.upload()
    assert len(uploaded) == 1, 'Upload exactly one reviewed engine source file.'
    name, data = next(iter(uploaded.items()))
    assert re.fullmatch(r'[0-9a-f]{64}', REVIEWED_UPLOAD_SHA256), 'Pin the reviewed upload SHA-256 first.'
    candidate_engine.write_bytes(data)
    expected_candidate_sha = REVIEWED_UPLOAD_SHA256
else:
    raise ValueError('CANDIDATE_MODE must be cap_only or upload')

assert sha256_file(candidate_engine) == expected_candidate_sha, sha256_file(candidate_engine)
source = candidate_engine.read_text()
assert source.count('def closure(seed_state: State, max_states: int = 216)') == 1
assert 'def closure(seed_state: State, max_states: int = 100)' not in source
print('Candidate:', candidate_engine)
print('Candidate SHA-256:', sha256_file(candidate_engine))


## Exact H0 closure audit

This audit uses only the sealed face geometry, exact trace multiplication, the Fierz H0 action, and breadth-first closure. It does **not** construct Gram matrices, resolvents, amplitudes, linked coefficients, or comparison targets.


In [ ]:
import importlib.util
from collections import Counter

module_name = 'workhouse_mce_colab_candidate'
sys.modules.pop(module_name, None)
spec = importlib.util.spec_from_file_location(module_name, candidate_engine)
engine = importlib.util.module_from_spec(spec)
sys.modules[module_name] = engine
spec.loader.exec_module(engine)

EXPECTED_DOUBLE = {1: 960, 2: 150, 8: 6}
EXPECTED_R2 = {1: 10, 2: 271, 4: 740, 5: 64, 6: 92, 8: 78, 13: 10, 24: 176, 216: 8}

def canonical_link_topology(state):
    relabel, occurrences = {}, []
    for link, is_u in state.occ:
        if link not in relabel:
            relabel[link] = len(relabel)
        occurrences.append((relabel[link], is_u))
    return engine.State(tuple(occurrences), state.part)

def singleton(vector):
    assert len(vector) == 1
    return next(iter(vector))

print('Candidate module imported target-blind.')


In [ ]:
started = time.monotonic()
patch, roots, _, _ = engine.build_o4_triality_candidate_full_t1_coverage()
assert len(patch.faces) == 93 and len(roots) == 3
builder = engine.ExactFaceInsertionBuilder(patch)
double_hist, closure_sizes, raw_r2 = Counter(), {}, 0

for root in roots.values():
    for root_sign in (-1, +1):
        root_state = builder._traces[(root, root_sign)]
        for first_face in range(len(patch.faces)):
            for first_sign in (-1, +1):
                first_seed = singleton(builder.insert_face({root_state: 1}, first_face, first_sign))
                first_closure = engine.closure(first_seed)
                double_hist[len(first_closure)] += 1
                for recoupled in first_closure:
                    for second_face in range(len(patch.faces)):
                        for second_sign in (-1, +1):
                            raw_r2 += 1
                            seed = singleton(builder.insert_face({recoupled: 1}, second_face, second_sign))
                            topology = canonical_link_topology(seed)
                            if topology not in closure_sizes:
                                closure_sizes[topology] = len(engine.closure(topology))

r2_hist = Counter(closure_sizes.values())
saturated = [state for state, size in closure_sizes.items() if size == 216]
cap215_failures = 0
for state in saturated:
    try:
        engine.closure(state, max_states=215)
    except engine.ExactEngineError:
        cap215_failures += 1
    else:
        raise AssertionError('cap 215 completed a saturated topology')

assert dict(sorted(double_hist.items())) == EXPECTED_DOUBLE
assert raw_r2 == 243_288
assert len(closure_sizes) == 1_449
assert dict(sorted(r2_hist.items())) == EXPECTED_R2
assert max(closure_sizes.values()) == 216
assert cap215_failures == len(saturated) == 8

closure_report = {
    'status': 'PASS',
    'method': 'geometry/state/Fierz/H0 BFS only',
    'forbidden_inputs_used': [],
    'patch_faces': 93,
    'roots': 3,
    'double_trace_histogram': dict(sorted(double_hist.items())),
    'raw_r2_seeds': raw_r2,
    'unique_r2_topologies': len(closure_sizes),
    'r2_closure_histogram': dict(sorted(r2_hist.items())),
    'maximum_closure': 216,
    'cap_215_failures': cap215_failures,
    'cap_216_completions': len(saturated),
    'elapsed_seconds': round(time.monotonic() - started, 3),
}
print(json.dumps(closure_report, indent=2, sort_keys=True))


In [ ]:
def run_engine(*args):
    proc = subprocess.run([sys.executable, str(candidate_engine), *args], capture_output=True, text=True)
    text = proc.stdout + proc.stderr
    if proc.returncode:
        print(text)
        raise RuntimeError(f'engine command failed: {args}')
    return text

selftest_text = run_engine('--self-test')
assert '47/47 exact Phase-2/Phase-3 gates passed' in selftest_text
assert '[FAIL]' not in selftest_text
print(selftest_text.splitlines()[-2])

preflight_text = run_engine('--geometry-preflight')
assert 'TRIALITY_CANDIDATE_PREFLIGHT_PASS_609_NO_PHYSICS' in preflight_text
payload = json.loads(preflight_text[preflight_text.index('{'):preflight_text.rindex('}') + 1])
assert payload['total_exact_cluster_evaluations'] == 609
assert payload['physics_contractions_run'] == 0
assert payload['candidate_coverage_certificate_sha256'] == COVERAGE_SHA256
assert payload['preflight_sha256'] == PREFLIGHT_SHA256
print('Geometry preflight: PASS 609, zero physics')
print('Coverage SHA-256:', payload['candidate_coverage_certificate_sha256'])
print('Preflight SHA-256:', payload['preflight_sha256'])


## G3 output-sufficiency gate

A successful cap audit is not permission to spend the 609-cluster run. The engine must checkpoint the exact fourth-order root-to-ket row and seal the raw endpoint ledger, 189-record kernel, exact shape block, and band points from the same authenticated run. The shape fit must not consume the blind R point.


In [ ]:
import inspect

checkpoint_source = inspect.getsource(engine.Phase3SQLiteCheckpoint)
seal_source = inspect.getsource(engine.issue_target_blind_m4_seal)
required_checkpoint = ('h4_endpoint_json',)
required_seal = ('raw_h4_endpoint_ledger', 'h4_kernel', 'h4_kernel_sha256', 'shape', 'band_points', 'shape_extraction')
missing_checkpoint = [x for x in required_checkpoint if x not in checkpoint_source]
missing_seal = [x for x in required_seal if x not in seal_source]
R_BLIND_DECLARED = (
    'R_used_in_fit' in seal_source
    and ('False' in seal_source or 'false' in seal_source)
)
G3_OUTPUT_READY = not missing_checkpoint and not missing_seal and R_BLIND_DECLARED

g3_output_report = {
    'status': 'PASS' if G3_OUTPUT_READY else 'FAIL_CLOSED',
    'missing_checkpoint_fields': missing_checkpoint,
    'missing_scientific_payload_fields': missing_seal,
    'blind_R_declared': R_BLIND_DECLARED,
    'production_authorized': False,
}
print(json.dumps(g3_output_report, indent=2, sort_keys=True))
if not G3_OUTPUT_READY:
    print('\nEXPECTED FOR CAP-ONLY ENGINE: closure is fixed, but production remains blocked.')


In [ ]:
validation_report = {
    'schema': 'WORKHOUSE-MCE-COLAB-RERUN-READINESS-v1',
    'frozen_commit': FROZEN_COMMIT,
    'base_engine_sha256': BASE_SHA256,
    'candidate_engine_sha256': sha256_file(candidate_engine),
    'closure_audit': closure_report,
    'self_tests': '47/47 PASS',
    'geometry_preflight': {
        'evaluations': 609,
        'physics_contractions': 0,
        'coverage_sha256': payload['candidate_coverage_certificate_sha256'],
        'preflight_sha256': payload['preflight_sha256'],
    },
    'g3_output_readiness': g3_output_report,
    'rerun_authorized': bool(G3_OUTPUT_READY),
    'target_inputs': [],
}
report_path = DERIVED / 'MCE_RERUN_READINESS_REPORT.json'
report_path.write_text(json.dumps(validation_report, indent=2, sort_keys=True))
print('Report:', report_path)
print('Report SHA-256:', sha256_file(report_path))

# Uncomment to download the derived engine and report.
# from google.colab import files
# files.download(str(candidate_engine))
# files.download(str(report_path))


## Optional sealed production stage — disabled by default

Only use these cells with a separately reviewed, SHA-pinned G3-capable engine for which `G3_OUTPUT_READY` is true. Google Drive is used for the authenticated SQLite checkpoint, resume secret, freeze, logs, and final certificate. Never inspect partial scientific values before the 609-row seal.


In [ ]:
AUTHORIZE_FREEZE = False
AUTHORIZE_609_CLUSTER_RUN = False
DRIVE_RUN_DIR = '/content/drive/MyDrive/WORKHOUSE_MCE_G3_REVISED'

if AUTHORIZE_FREEZE or AUTHORIZE_609_CLUSTER_RUN:
    assert G3_OUTPUT_READY, 'G3 endpoint/kernel output contract is not satisfied.'
    assert CANDIDATE_MODE == 'upload', 'Production requires a separately reviewed G3-capable upload.'
    from google.colab import drive
    drive.mount('/content/drive')
    run_dir = Path(DRIVE_RUN_DIR)
    run_dir.mkdir(parents=True, exist_ok=True)
else:
    print('Production remains disabled.')


In [ ]:
if AUTHORIZE_FREEZE:
    freeze_cmd = [sys.executable, str(harness), '--engine', str(candidate_engine), 'freeze']
    subprocess.run(freeze_cmd, cwd=run_dir, check=True)
    print('Freeze complete. Review FREEZE.json and logs before enabling production.')
else:
    print('Freeze cell skipped.')


In [ ]:
if AUTHORIZE_609_CLUSTER_RUN:
    assert AUTHORIZE_FREEZE, 'Run the freeze cell first in this session.'
    assert (run_dir / 'FREEZE.json').is_file()
    run_cmd = [
        sys.executable, str(harness), '--engine', str(candidate_engine),
        '--output', str(run_dir / 'HODGE_SU3_EXACT_MARKED_CLUSTER_M4_CERTIFICATE.json'),
        '--checkpoint', str(run_dir / 'HODGE_SU3_EXACT_MARKED_CLUSTER_M4_CHECKPOINT.sqlite'),
        'run',
    ]
    print('Launching the sealed resumable sweep. Interrupting retains the authenticated checkpoint.')
    subprocess.run(run_cmd, cwd=run_dir, check=True)
else:
    print('609-cluster production cell skipped.')
